第12章

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler  #標準化
from sklearn.linear_model import LogisticRegression  #ロジスティク回帰

In [2]:
df=pd.read_csv('../sukkiri-ml2-codes/datafiles/iris.csv')
display(df.head(3))

,がく片長さ,がく片幅,花弁長さ,花弁幅,種類
0,0.22,0.63,0.08,0.04,Iris-setosa
1,0.17,0.42,0.35,0.04,Iris-setosa
2,0.11,0.50,0.13,0.04,Iris-setosa


In [3]:
#欠損値を補完
df_mean=df.mean(numeric_only=True)
train2=df.fillna(df_mean)

#x,yに分割
x=train2.loc[:,:'花弁幅']
y=train2['種類']

#標準化
sc=StandardScaler()
new=sc.fit_transform(x)

#訓練データと検証データに分割（テストデータへの分割は省略するみたい）
x_train,x_val,y_train,y_val=train_test_split(new,y,test_size=0.2,random_state=0)

In [4]:
model=LogisticRegression(random_state=0,C=0.1,multi_class='auto',solver='lbfgs')

model.fit(x_train,y_train)
print(model.score(x_train,y_train))
print(model.score(x_val,y_val))

0.8666666666666667
0.8333333333333334


c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [5]:
#係数の確認
model.coef_

array([[-0.53168668,  0.48573155, -0.52638431, -0.83213317],
       [ 0.09482289, -0.44707757, -0.00107471, -0.04407356],
       [ 0.4368638 , -0.03865398,  0.52745902,  0.87620673]])

In [6]:
#新規データでの予測
x_new=[[1,2,3,4]]
model.predict(x_new)

array(['Iris-virginica'], dtype=object)

In [7]:
model.predict_proba(x_new)

array([[4.02790964e-05, 3.02983301e-03, 9.96929888e-01]])

ランダムフォレストの実装

In [8]:
df=pd.read_csv('../sukkiri-ml2-codes/datafiles/Survived.csv')
display(df.head(3))

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [ ]:
#ageが欠損している行の補完
def fill_age(df,pclass,survived,value):
    joken1 = (df['Pclass']==pclass)
    joken2 = (df['Survived']==survived)
    joken3 = df['Age'].isnull()
    df.loc[(joken1)&(joken2)&(joken3),'Age']=value
    return df

print(df.pivot_table(index='Pclass',columns='Survived',values='Age'))

Survived          0          1
Pclass                        
1         43.695312  35.368197
2         33.544444  25.901566
3         26.555556  20.646118


In [10]:
#ピボットテーブルの値を参考に補完
df=fill_age(df,1,0,43)
df=fill_age(df,1,1,35)
df=fill_age(df,2,0,33)
df=fill_age(df,2,1,25)
df=fill_age(df,3,0,26)
df=fill_age(df,3,1,20)

In [13]:
#特徴量として利用する列のリスト
col=['Pclass','Age','SibSp','Parch','Fare']

x=df[col]
y=df['Survived']

#sex列は文字の列なのでダミー変数化
dummy=pd.get_dummies(df['Sex'],drop_first=True,dtype=int)
x=pd.concat([x,dummy],axis=1)
x.head(2)

,Pclass,Age,SibSp,Parch,Fare,male
0,3,22.0,1,0,7.2500,1
1,1,38.0,1,0,71.2833,0


In [ ]:
#ランダムフォレストのインポート
from sklearn.ensemble import RandomForestClassifier
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=0)
model=RandomForestClassifier(n_estimators=200,random_state=0)

In [17]:
model.fit(x_train,y_train)

print(model.score(x_train,y_train))
print(model.score(x_test,y_test))

0.9859550561797753
0.8770949720670391


In [ ]:
depths = [tree.get_depth() for tree in model.estimators_]

print("最小の深さ:", min(depths))
print("最大の深さ:", max(depths))
print("平均の深さ:", sum(depths) / len(depths))

'''
ランダムフォレストでは、各木が過学習していても、複数の木を総合することで過学習傾向を抑えられる。
ゆえにmax_depth=Noneでも問題がない。（もちろん分類木でこれをすると悲惨なことになる）
'''

最小の深さ: 12
最大の深さ: 24
平均の深さ: 16.385


In [ ]:
importance=model.feature_importances_  #特徴量重要度
pd.Series(importance,index=x_train.columns)  #シリーズにして表示

Pclass    0.080805
Age       0.298318
SibSp     0.048996
Parch     0.034688
Fare      0.282337
male      0.254856
dtype: float64